In [ ]:
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import *
from delta.tables import DeltaTable

MINIO_ACCESS_KEY = "pkIeKAn4xpjoOgXiHQPw"
MINIO_SECRET_KEY = "JZRBfRszxLZzPeaAQaEXk32JxUKv25DVjUoO06Rk"
DATABASE = "default"
BUCKET_BRONZE = "bronze"
BUCKET_SILVER = "silver"
BUCKET_GOLD = "gold"

In [ ]:
%%time
spark = SparkSession.builder.master("spark://spark-master:7077") \
    .appName("MyAppM6Class02") \
    .config("spark.eventLog.enabled", "true") \
    .config("spark.eventLog.dir", "file:/home/jovyan/work/spark-logs") \
    .config("spark.history.fs.logDirectory", "file:/home/jovyan/work/spark-logs") \
    .config("log4j.rootCategory", "INFO, console") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "1536m") \
    .config("spark.driver.memory", "1536m") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.storage.memoryFraction", "0.4") \
    .config("spark.shuffle.memoryFraction", "0.5") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "512m") \
    .config("spark.sql.parquet.compression.codec", "gzip") \
    .config("spark.sql.orc.compression.codec", "zlib") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC") \
    .config("spark.cleaner.referenceTracking.cleanCheckpoints", "true") \
    .config("spark.executor.cleanupOnShutdown", "true") \
    .getOrCreate()

In [ ]:
sc = spark.sparkContext
hadoop_conf = sc._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
hadoop_conf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

In [ ]:
table_gold = "auditing_finance_fake"
location_gold = f"s3a://{BUCKET_GOLD}/delta/{table_gold}"

# JOB OF INGESTION

## 1. Define the Schema of the Table

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
from pyspark.sql import SparkSession

# Initialize Spark session (already available in Databricks)
spark = SparkSession.builder.getOrCreate()

# Define the schema for the financial table
schema_financial = StructType([
    StructField("id", IntegerType(), nullable=False),
    StructField("account", StringType(), nullable=False),
    StructField("transaction_type", StringType(), nullable=False),
    StructField("amount", DoubleType(), nullable=False),
    StructField("transaction_date", TimestampType(), nullable=False)
])


## 2. Create 10 Fake Records

In [ ]:
from datetime import datetime

# Create a list of 10 fake records
fake_data = [
    (1, "Account_A", "Deposit", 1000.0, datetime.now()),
    (2, "Account_B", "Withdrawal", 200.0, datetime.now()),
    (3, "Account_C", "Transfer", 500.0, datetime.now()),
    (4, "Account_D", "Payment", 300.0, datetime.now()),
    (5, "Account_E", "Deposit", 1500.0, datetime.now()),
    (6, "Account_F", "Withdrawal", 100.0, datetime.now()),
    (7, "Account_G", "Transfer", 750.0, datetime.now()),
    (8, "Account_H", "Payment", 400.0, datetime.now()),
    (9, "Account_I", "Deposit", 2000.0, datetime.now()),
    (10, "Account_J", "Withdrawal", 50.0, datetime.now())
]

# Create DataFrame with fake records
df_initial = spark.createDataFrame(fake_data, schema=schema_financial)


In [ ]:
df_initial.show(truncate=False)

## 3. Write the Data into a Delta Table

In [ ]:
%%time
(
    df_initial.write.format("delta")
    .mode("overwrite")
    .save(location_gold)
)

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE}.{table_gold}
    USING DELTA
    LOCATION '{location_gold}'
""")

## 4. Simulate Operations: UPDATE, DELETE, INSERT, MERGE, RESTORE, OPTMIZE
First, load the Delta table:

In [ ]:
# Load the existing Delta table
delta_table = DeltaTable.forPath(spark, location_gold)


### Simulate an UPDATE:

In [ ]:
# For example, increase the amount of transactions in Account_A by 10%
delta_table.update(
    condition = "account = 'Account_A'",
    set = { "amount": "amount * 1.20" }
)

### Simulate a DELETE:

In [ ]:
# Delete records where amount is less than 100
delta_table.delete("amount < 100")

### Simulate an INSERT:

In [ ]:
# Create new records to insert
new_data = [
    (11, "Account_K", "Deposit", 600.0, datetime.now()),
    (12, "Account_L", "Withdrawal", 300.0, datetime.now())
]
df_new = spark.createDataFrame(new_data, schema=schema_financial)

# Insert new records into the Delta table
df_new.write.format("delta").mode("append").save(location_gold)

### Simulate a MERGE:

In [ ]:
# Create a DataFrame with records to merge
merge_data = [
    (3, "Account_C", "Transfer", 550.0, datetime.now()),  # Update existing value
    (13, "Account_M", "Payment", 800.0, datetime.now())   # New record
]
df_merge = spark.createDataFrame(merge_data, schema=schema_financial)

# Execute merge: update if id exists, insert if not
delta_table.alias("tgt").merge(
    source = df_merge.alias("src"),
    condition = "tgt.id = src.id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()


### Optmize table

In [ ]:
spark.sql(f"OPTIMIZE delta.`{location_gold}`")

### Perform a Restore Operation

In [ ]:
# Consultar o histórico para escolher uma versão anterior
history_df = delta_table.history().orderBy("version", ascending=False)
history_df.show(5, truncate=False)

# Escolha uma versão anterior para restaurar (por exemplo, versão 1)
version_to_restore = 0

# Executar a operação de restauração para a versão escolhida
delta_table.restoreToVersion(version_to_restore)
print(f">> Table {table_gold} it was restored successfully to version {version_to_restore}!")

<hr style="border: 2px solid green;">

# JOB OF AUDITING

## 5. Extract Data from the History
After performing the operations, extract a new DataFrame with audit information:

> Note: Some columns like userName, userMetadata, operationParameters.engine may not be present or may be null depending on your environment and how operations were executed. Adjust accordingly based on available metadata.

In [ ]:
history_df = delta_table.history()
history_df.show(truncate=False)

In [ ]:
# +-------+-------------------+------+--------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+-----------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
# |version|timestamp          |userId|userName|operation|operationParameters                                                                                                                                                         |job |notebook|clusterId|readVersion|isolationLevel   |isBlindAppend|operationMetrics                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        |userMetadata|engineInfo                         |
# +-------+-------------------+------+--------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+-----------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
# |6      |2025-01-20 16:32:42|null  |null    |RESTORE  |{version -> 0, timestamp -> null}                                                                                                                                           |null|null    |null     |5          |Serializable     |false        |{numRestoredFiles -> 2, removedFilesSize -> 1833, numRemovedFiles -> 1, restoredFilesSize -> 3480, numOfFilesAfterRestore -> 2, tableSizeAfterRestore -> 3480}                                                                                                                                                                                                                                                                                                                                                                          |null        |Apache-Spark/3.4.1 Delta-Lake/2.4.0|
# |5      |2025-01-20 16:32:25|null  |null    |OPTIMIZE |{predicate -> [], zOrderBy -> []}                                                                                                                                           |null|null    |null     |4          |SnapshotIsolation|false        |{numRemovedFiles -> 4, numRemovedBytes -> 6873, p25FileSize -> 1833, numDeletionVectorsRemoved -> 0, minFileSize -> 1833, numAddedFiles -> 1, maxFileSize -> 1833, p75FileSize -> 1833, p50FileSize -> 1833, numAddedBytes -> 1833}                                                                                                                                                                                                                                                                                                     |null        |Apache-Spark/3.4.1 Delta-Lake/2.4.0|
# |4      |2025-01-20 16:32:19|null  |null    |MERGE    |{predicate -> ["(id#557 = id#2207)"], matchedPredicates -> [{"actionType":"update"}], notMatchedPredicates -> [{"actionType":"insert"}], notMatchedBySourcePredicates -> []}|null|null    |null     |3          |Serializable     |false        |{numTargetRowsCopied -> 4, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 1764, numTargetBytesRemoved -> 1744, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 4196, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, scanTimeMs -> 2825, numTargetRowsUpdated -> 1, numOutputRows -> 6, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1266}|null        |Apache-Spark/3.4.1 Delta-Lake/2.4.0|
# |3      |2025-01-20 16:32:11|null  |null    |WRITE    |{mode -> Append, partitionBy -> []}                                                                                                                                         |null|null    |null     |2          |Serializable     |true         |{numFiles -> 3, numOutputRows -> 2, numOutputBytes -> 4065}                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |null        |Apache-Spark/3.4.1 Delta-Lake/2.4.0|
# |2      |2025-01-20 16:32:08|null  |null    |DELETE   |{predicate -> ["(amount#560 < 100.0)"]}                                                                                                                                     |null|null    |null     |1          |Serializable     |false        |{numRemovedFiles -> 1, numRemovedBytes -> 1736, numCopiedRows -> 4, numAddedChangeFiles -> 0, executionTimeMs -> 3245, numDeletedRows -> 1, scanTimeMs -> 2527, numAddedFiles -> 1, numAddedBytes -> 1721, rewriteTimeMs -> 717}                                                                                                                                                                                                                                                                                                        |null        |Apache-Spark/3.4.1 Delta-Lake/2.4.0|
# |1      |2025-01-20 16:32:00|null  |null    |UPDATE   |{predicate -> ["(account#558 = Account_A)"]}                                                                                                                                |null|null    |null     |0          |Serializable     |false        |{numRemovedFiles -> 1, numRemovedBytes -> 1744, numCopiedRows -> 4, numAddedChangeFiles -> 0, executionTimeMs -> 5228, scanTimeMs -> 4374, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1744, rewriteTimeMs -> 846}                                                                                                                                                                                                                                                                                                        |null        |Apache-Spark/3.4.1 Delta-Lake/2.4.0|
# |0      |2025-01-20 16:31:46|null  |null    |WRITE    |{mode -> Overwrite, partitionBy -> []}                                                                                                                                      |null|null    |null     |null       |Serializable     |false        |{numFiles -> 2, numOutputRows -> 10, numOutputBytes -> 3480}                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |null        |Apache-Spark/3.4.1 Delta-Lake/2.4.0|
# +-------+-------------------+------+--------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+-----------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+

In [ ]:
# Select and rename columns for auditing
audit_history = history_df.selectExpr(
    "operation as Operation",
    "timestamp as Date",
    "userName as User",               # May vary depending on how user info is captured
    "userMetadata as Job",            # Additional info if available
    "engineInfo as Engine",  # Example column for engine, adjust as needed
    "operationMetrics.numUpdatedRows as UpdatedRows",
    "operationMetrics.numOutputRows as InsertedRows",
    "operationMetrics.numDeletedRows as DeletedRows"
)

audit_history.show(truncate=False)

## 6. Create an Audit Table for Future Updates

Create or update a separate Delta table to store the extracted audit logs, which will be fed as new operations occur:

In [ ]:
table_gold_audit = f"{table_gold}_operation"
location_gold_audit = f"s3a://{BUCKET_GOLD}/delta/{table_gold_audit}"

In [ ]:
# Write the audit DataFrame to a Delta table (overwrite mode for initialization)
audit_history.write.format("delta").mode("overwrite").save(location_gold_audit)

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE}.{table_gold_audit}
    USING DELTA
    LOCATION '{location_gold_audit}'
""")

## 7. Update the Audit Table with New Operations (Example)
To continuously feed the audit table with new operations, you can schedule a job or use triggers to:

- Extract recent operations from the main table.

- Insert or update records in the audit table.

A basic periodic update example:

In [ ]:
# Retrieve the complete history of the table
delta_table = DeltaTable.forPath(spark, location_gold)
history_df = delta_table.history()

# Get the last audited version
last_audited_version = history_df.agg({"version": "max"}).collect()[0][0]

# Extract new operations after the last audited version
new_operations_df = delta_table.history().filter(f"version = {last_audited_version}")

# Process and select the desired fields
new_audit_operations = new_operations_df.selectExpr(
    "operation as Operation",
    "timestamp as Date",
    "userName as User",
    "userMetadata as Job",
    "engineInfo as Engine",
    "operationMetrics.numUpdatedRows as UpdatedRows",
    "operationMetrics.numOutputRows as InsertedRows",
    "operationMetrics.numDeletedRows as DeletedRows"
)

# Append new operations to the audit table
new_audit_operations.write.format("delta").mode("append").save(location_gold_audit)

In [ ]:
spark.sql(f"SELECT * FROM {DATABASE}.{table_gold_audit} ORDER BY Date DESC").show(truncate=False)

In [ ]:
spark.sql(f"SELECT * FROM {DATABASE}.{table_gold_audit} ORDER BY Date DESC").show(truncate=False)